In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np #专门做数字计算的工具包，全名 Numerical Python

path='/content/drive/MyDrive/ai_project/adults-media-literacy-2025-respondent-level-data.csv'
df=pd.read_csv(path, na_values=' ')

sample=df[(df['MODULEA']==1)&(df['INTUSED']==1)].copy()

#复印一份独立的新表，只包含有效样本，下面的清洗都是在sample上增加改动改它不会影响 df

sample.shape

Mounted at /content/drive


(3515, 566)

In [2]:
sample['Y1']=sample['A6'].map({1:3,2:2,3:1,4:0}) #在 sample 里新建一列，名字叫 Y1。左边写一个还不存在的列名，pandas 就会新建它

                                                 #{} dictionary（字典）, 查key 输出value--构造 Y1（A6 重新编码）

                                                 #.map()将指定的函数依次作用于一个或多个可迭代对象（如列表、字典等）的每个元素上，并返回一个包含结果的迭代器

                                                 # .map() Series列 method 一列里的每一个值，把它当成 key 去字典里查，然后用查到的 value 替换它。

                                                 # 字典里没有5, .map() 查不到，就自动变成 NaN

sample['Y1'].value_counts(dropna=False).sort_index()#让 value_counts() 把 NaN 也数出来（默认是不数的--dropna=True)

                                                    #value_counts() 默认按人数从多到少排。.sort_index() 是按行名（这里就是 0、1、2、3）从小到大排

                                                    #按值从小到大排（0、1、2、3），而不是按人数多少排，方便看

,count
Y1,
0.0,525
1.0,1156
2.0,893
3.0,604
NaN,337


In [3]:
#.map()底层逻辑
#def change_value(x):
#    if x == 1: return 3
#    elif x == 2: return 2
#    elif x == 3: return 1
#    elif x == 4: return 0
#    else: return None  # 其他找不到的变空值

In [4]:
sample['seen_sumary']=(sample['A6']!=5).astype(int)#你的 A6 不是 5 吗？不是 5 → True（见过），是 5 → False（没见过）

                                                   #A6如果是Na!=5-True,没被提问会被当作见过，但sample里没有NaN,df才有

                                                   #astype = “as type”，换成某种类型。int 是整数。True 变成 1，False 变成 0

                                                   #1/0 以后放进模型、算比例更方便（比如直接 .mean() 就是"见过的人占多少"）
sample['seen_sumary'].value_counts()

,count
seen_sumary,
1,3178
0,337


In [5]:
sample['income_dk']=(sample['E9']==8).astype(int)
sample['income_dk'].value_counts()

,count
income_dk,
0,2788
1,727


In [6]:
(sample['SUMAI']==3).value_counts()


,count
SUMAI,
False,3274
True,241


In [9]:
sample['A4_c']=sample['A4'].fillna(0)
sample['A4_c'].value_counts(dropna=False).sort_index()

,count
A4_c,
0.0,241
1.0,246
2.0,876
3.0,1841
4.0,311


In [10]:
tab=pd.crosstab(sample['SUMAI'],sample['A4_c'].fillna(0))
tab

A4_c,0.0,1.0,2.0,3.0,4.0
SUMAI,,,,,
1.0,0,214,713,1014,119
2.0,0,32,163,827,192
3.0,241,0,0,0,0


Check: A4_c = 0 only for respondents unaware of AI (SUMAI = 3)

In [12]:
sample['A5_c']=sample['A5'].fillna(0)
sample['A5_c'].value_counts(dropna=False).sort_index()

,count
A5_c,
0.0,241
1.0,348
2.0,1144
3.0,935
4.0,573
5.0,214
6.0,60


In [13]:
tab1=pd.crosstab(sample['SUMAI'],sample['A5_c'].fillna(0))
tab1

A5_c,0.0,1.0,2.0,3.0,4.0,5.0,6.0
SUMAI,,,,,,,
1.0,0,289,864,545,284,60,18
2.0,0,59,280,390,289,154,42
3.0,241,0,0,0,0,0,0


In [14]:
sample.shape

(3515, 571)

In [15]:
out_path = '/content/drive/MyDrive/ai_project/sample_clean_v1.csv'#out是输出，和path区分，v1第一版
sample.to_csv(out_path, index=False)
#不要把行名存进去。不写的话，pandas 会把行名（行号 3、7、8……）当成多出来的一列存进文件，下次读进来就会多一列没名字的东西

In [16]:
check = pd.read_csv(out_path)#检查
check.shape

(3515, 571)

## Cleaning part 2

In [18]:
a3_cols=['A3A','A3B','A3C','A3D','A3E','A3F','A3G','A3H']

In [22]:
sample[a3_cols].head(10)

,A3A,A3B,A3C,A3D,A3E,A3F,A3G,A3H
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
23,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Y2 是怎么算出来的（4 个人举例）

**① `sample[a3_cols]`：从大表里切出 8 列**

| 行名 | SUMAI | A3A | A3B | A3C | A3D | A3E | A3F | A3G | A3H | 说明 |
|---|---|---|---|---|---|---|---|---|---|---|
| 1 | 1 | 1 | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 用过 AI，选了 2 项 |
| 7 | 2 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 没用过（Ofcom 填的 0） |
| 9 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 用过，一项没选（那 36 人之一） |
| 13 | 3 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 没听说过 AI（Ofcom 填的 0） |

In [27]:
sample[a3_cols].sum(axis=1)#.sum()和.sum(axis=0)都是一列往下相加，算得是每个活动有多少人选,
                           #.sum(axis=1)是一行相加，每个人选了几种-Y2预测ai广度用

,0
1,0.0
7,0.0
9,0.0
13,0.0
15,1.0
...,...
7521,2.0
7523,0.0
7524,1.0
7525,1.0


**② `.sum(axis=1)`：每行横着加，8 列压成 1 列**

In [28]:
sample[a3_cols].sum(axis=1).value_counts().sort_index()#ofcom数据把sumai=2，3没用过ai也标0，包含一部分没用过ai要当空值，一部分填a3i=其他用途，a3j不知道，这两个是应该保留的
                                                       #上个列表算出来是每行=每个人8个算了几个
                                                       #value_counts 每一个唯一值出现次数，1491就是一共有1491人选0
                                                       #sort按标签，不按人数多少

,count
0.0,1491
1.0,555
2.0,484
3.0,395
4.0,249
5.0,163
6.0,108
7.0,41
8.0,29


In [46]:
((sample['SUMAI']==2)|(sample['SUMAI']==3)).value_counts()#1455就是没用过ai的人，应该要当空值处理
#|或者，&并且
#(sample['SUMAI'] != 1).value_counts()同样效果
#sample['SUMAI'].isin([2, 3]).value_counts()，.isin()："是不是在这个 list 里面"：

,count
SUMAI,
False,2060
True,1455


**③ `no_aiusers = (SUMAI==2) | (SUMAI==3)`：每个人一张 True/False 牌**

| 行名 | SUMAI | no_aiusers |
|---|---|---|
| 1 | 1 | False |
| 7 | 2 | **True** |
| 9 | 1 | False |
| 13 | 3 | **True** |

In [51]:
no_aiusers=((sample['SUMAI']==2)|(sample['SUMAI']==3))

In [52]:
zero_users = sample[(sample['SUMAI'] == 1) & (sample[a3_cols].sum(axis=1) == 0)]#1491-1455=36是满足用过ai，ai用处填0的有36人，
zero_users.shape

(36, 572)

In [53]:
pd.crosstab(zero_users['A3I'],zero_users['A3J'])#填a3j=17人是不知道，a3i=19人是其他用途，不再前面8种里

A3J,0.0,1.0
A3I,,
0.0,0,17
1.0,19,0


In [54]:
sample['Y2']=(sample[a3_cols].sum(axis=1))
sample.loc[no_aiusers,'Y2']=np.nan
sample['Y2'].value_counts(dropna=False).sort_index()

,count
Y2,
0.0,36
1.0,555
2.0,484
3.0,395
4.0,249
5.0,163
6.0,108
7.0,41
8.0,29


**④ `sample['Y2'] = ...`：贴回大表，叫 Y2**

| 行名 | SUMAI | Y2 | 说明 |
|---|---|---|---|
| 1 | 1 | 2 | |
| 7 | 2 | 0 | 假的 0 |
| 9 | 1 | 0 | 真的 0 |
| 13 | 3 | 0 | 假的 0 |

**⑤ `sample.loc[no_aiusers, 'Y2'] = np.nan`：True 的行，把 Y2 改成空**

| 行名 | no_aiusers | Y2 改之前 | Y2 改之后 |
|---|---|---|---|
| 1 | False | 2 | 2（不动） |
| 7 | True | 0 | **NaN** |
| 9 | False | 0 | 0（真的 0 保留） |
| 13 | True | 0 | **NaN** |


##下一步，x2其中（上网活动数）

In [55]:
activity_cols = ['Q13A', 'Q13B', 'Q13C', 'Q13D', 'Q13E', 'Q13F', 'Q13G', 'Q13H', 'Q13I', 'Q13J', 'Q13K', 'Q13L', 'Q13M']

# 计算每个人在 Q13A–Q13M 中选择了多少个活动，并存入新列 activity_count
sample['activity_count'] = sample[activity_cols].sum(axis=1)

# 检查新列的分布
sample['activity_count'].value_counts(dropna=False).sort_index()

,count
activity_count,
0,54
1,74
2,166
3,206
4,291
5,317
6,407
7,386
8,375


In [56]:
pd.crosstab(sample['BREADTH'],sample['activity_count'])

activity_count,0,1,2,3,4,5,6,7,8,9,10,11,12,13
BREADTH,,,,,,,,,,,,,,
1,54,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,74,166,206,291,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,317,407,386,375,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,371,308,239,185,136


In [57]:
v2_path = '/content/drive/MyDrive/ai_project/sample_clean_v2.csv'
sample.to_csv(v2_path, index=False)

check = pd.read_csv(v2_path)
check.shape

(3515, 573)